<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# 📘 EarthDaily Agriculture - Emergence Processor Bulk Extraction

## **✅ Step 1: Initialisation**

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
import os
manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")

## **🛠️ Step 2: Get entities**

### Option 1 - Load entities from Earthdaily platform

In [ ]:
manager.load_seasonfields()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 2 - Load entities from Earthdaily platform (Batch method - Recommended for large datasets)

In [ ]:
# This method uses batch processing to efficiently load 5000 entities

# Load all available entities using batch processing
manager.load_seasonfields_batch()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 3 -Load entities from file

In [ ]:
from earthdaily.agriculture.core.geometry import load_geodataframe

# Load entities from an external file (supports .shp, .parquet, .gpq, .geojson, .json, .gpkg, .csv)
file_path = "inputs/agroterrint.parquet"
manager.sfd_list = load_geodataframe(file_path, verbose=True)
print(manager.sfd_list.head())

## **📥 Step 3: Extract analytics - Debug function from processor_emergence_functions.py**

### 🗺️ Configure extraction

In [ ]:
# Import your class
from earthdaily.agriculture.processors.processor_emergence_functions import EmergenceExtractor
extractor = EmergenceExtractor(manager.bearer_token, manager.token_expiration,config=manager.config )

# Define column mapping to match DataFrame column names from the platform
column_mapping = {"crop": "crop.id"}

extractor.setup_emergence_parameters(
                                    emergence_type="HISTORICAL",
                                    season_duration=120,
                                    season_start_day=1,
                                    season_start_month=4,
                                    year='2025',
                                    data_source='LR',
                                    publish_af= True,
                                    partial_frequency=50,
                                    column_mapping=column_mapping
                                    )

### 🗺️ Test functions

In [ ]:
# Prepare test seasonfield_data
seasonfield_data = {
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop": "CORN"
}

#### Test get_emergence_api

In [ ]:
print("\n--- Test: get_emergence ---")
# Invoke the function
try:
    result = extractor.get_emergence_api(seasonfield_data)
    print("✅ Raw API response received:")
    print(result if isinstance(result, dict) else result[:500])
except Exception as e:
    print(f"❌ API call failed: {e}")
    


#### Test get_inseason_monitoring_api_safe

In [ ]:
print("\n--- Test: get_emergence_safe ---")
safe_result = extractor.get_emergence_api_safe(seasonfield_data)
print(safe_result)

In [ ]:
print("\n--- Test: format_emergence_json ---")
if safe_result["success"] and safe_result["data"]:
    formatted_df = extractor.format_emergence_json(safe_result["data"])
    print(f"✅ Formatted DataFrame with {len(formatted_df)} rows:")
    display(formatted_df.head())
else:
    print("⚠️ Skipping format_emergence_json: No valid data from API.")

#### Showcase — HISTORICAL `historical_seasons` (matching-season filter)

For `emergence_type="HISTORICAL"` you can pass, **per field**, the set of calendar years that
field actually grew the target crop (its "matching seasons"). When supplied, the extractor:

1. keeps `emergence_year_N` only for those years and nulls the rest, and
2. emits a new `avg_emergence_matching_years` column — the average emergence recomputed over
   the retained years (as `MM-DD`) — while leaving `historical_average_emergence` (the raw API
   average over all 5 seasons) **unchanged**.

`emergence_year_N` maps to calendar year `year - N` (here `year=2025` → year_1=2024 … year_5=2020).
The season set accepts either a comma-separated string (`"2024,2022,2020"`) or a list of ints
(`[2024, 2022, 2020]`). In a bulk run it flows in from the entity row via `column_mapping`, exactly
like `crop`.

In [ ]:
print("\n--- Showcase: HISTORICAL with vs. without historical_seasons ---")
if safe_result["success"] and safe_result["data"]:
    # 1) Baseline — no matching seasons supplied: no filtering.
    #    avg_emergence_matching_years is null; historical_average_emergence is the raw API value.
    df_all = extractor.format_emergence_json(safe_result["data"])
    print("Without historical_seasons (all 5 seasons kept):")
    display(df_all)

    # 2) With a per-field season set — keep only the years this field grew the crop.
    #    Off-years are nulled and avg_emergence_matching_years is recomputed over the rest.
    #    Both input forms are accepted:
    matching_seasons_str = "2024,2022,2020"          # comma-separated string
    matching_seasons_list = [2024, 2022, 2020]        # list of ints (equivalent)

    df_matching = extractor.format_emergence_json(
        safe_result["data"], historical_seasons=matching_seasons_str
    )
    print(f"\nWith historical_seasons={matching_seasons_str!r} (off-years nulled, average recomputed):")
    display(df_matching)

    # Confirm the two input forms agree, and that the raw API average is untouched.
    df_list = extractor.format_emergence_json(
        safe_result["data"], historical_seasons=matching_seasons_list
    )
    same_avg = df_matching["avg_emergence_matching_years"].iloc[0] == df_list["avg_emergence_matching_years"].iloc[0]
    print(f"\nString and list forms agree: {same_avg}")
    print(f"historical_average_emergence unchanged: "
          f"{df_all['historical_average_emergence'].iloc[0]} == {df_matching['historical_average_emergence'].iloc[0]}")

    # In a bulk run, wire the season set in per field via column_mapping, e.g.:
    #   column_mapping={"crop": "crop.id", "historical_seasons": "matching_seasons"}
    # where the "matching_seasons" column holds "2024,2022,2020" (or [2024, 2022, 2020]) per row.
else:
    print("⚠️ Skipping showcase: No valid data from API.")

#### Test format_inseason_json

### 🗺️ process_single_entity_emergence

In [ ]:
import pandas as pd
row = pd.Series({
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop":"OTHERS"
})

result = extractor.process_single_entity_emergence(row)

print(result)

### 🗺️ process_emergence_bulk_extraction_parallel

In [ ]:
top25 = manager.sfd_list.head(30)
print(top25.columns)
# Launch extraction with 10 threads 
result = extractor.process_emergence_bulk_extraction_parallel(
    entity_list=top25,
    params=None,
    max_workers=20,
    output_path=manager.output_result_dir,
    fail_safe=False,
    filter_column="crop.id",
    filter_value="CORN",
    filter_type="exclude", # filter type used to 'include' or 'exclude' row matching column and value filter
    merge_existing='auto',
    skip_export=False,
    prefix="emergence"
)

print(f"\nResults summary:")
print(f"Total: {result['total_calculations']}")
print(f"Success: {result['successful_calculations']}")
print(f"Failed: {result['failed_calculations']}")
print(result["results_df"].columns)
print("\n🔍 First 3 errors:")
for i, error in enumerate(result['global_errors'][:3]):
    print(f"\nError {i+1}:")
    for key, value in error.items():
        print(f"  {key}: {value}")

In [ ]:
print(result["results_df"])

In [ ]:
# Get the clean DataFrame
results=result["results_df"]
print(results.columns)